In [0]:

import time

notebooks = [
    "./gold.dim_customers",
    "./gold.dim_products",
    "./gold.dim_sellers",
    "./gold.dim_geography",
    "./gold.dim_date",
    "./gold.fact_sales"
]

print("Starting Gold Layer Pipeline...")

start_all = time.time()

for nb in notebooks:

    try:

        start_nb = time.time()

        print(f"Running {nb}")

        dbutils.notebook.run(nb, timeout_seconds=0)

        duration = round(time.time() - start_nb, 2)

        print(f"SUCCESS: {nb} ({duration}s)")

    except Exception as e:

        print(f"ERROR: {nb} failed")
        print(str(e))

        raise e


# FINAL DATA VALIDATION

print("\nRunning Revenue Validation...")

silver_rev = spark.table("workspace.silver.order_items") \
    .selectExpr("sum(price)") \
    .collect()[0][0]

gold_rev = spark.table("workspace.gold.fact_sales") \
    .selectExpr("sum(sales_amount)") \
    .collect()[0][0]

if round(silver_rev,2) == round(gold_rev,2):

    print(f"DATA VALIDATED: Silver Revenue ({silver_rev}) matches Gold Revenue")

else:

    print(f"REVENUE MISMATCH: Silver ({silver_rev}) vs Gold ({gold_rev})")

print(f"\nTotal Pipeline Time: {round(time.time()-start_all,2)} seconds")